In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 100
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2

/home/ruanjohn/miniconda3/envs/mava-jax5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = decay_kappas[None, :, None, None]

In [3]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros((bsz, retnet_num_heads, retnet_embed_dim//retnet_num_heads, retnet_embed_dim//retnet_num_heads))
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [ ]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
)

TypeError: cannot reshape array of shape (16, 400, 400) (size 2560000) into shape (16, 400, 1) (size 6400)

: 

In [ ]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):

    # todo: reset later
    hstate = hstate * decay_kappas
    obs_i = obs[:, step*num_agents:(step+1)*num_agents, ...]
    dones_i = dones[:, step*num_agents:(step+1)*num_agents]
    step_counts_i = step_counts[:, step*num_agents:(step+1)*num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, method="recurrent")
    act_output.append(out)

In [ ]:
act_output = jnp.concatenate(act_output, axis=1)

In [ ]:
act_output.shape

(16, 400, 32)

In [ ]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts)

In [ ]:
train_out.shape

(16, 400, 32)

In [ ]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(0.0125015, dtype=float32)

In [ ]:
jnp.abs(train_out - act_output)

Array([[[0.00123617, 0.01106529, 0.01473209, ..., 0.00156621,
         0.00750967, 0.00789852],
        [0.00068307, 0.04560087, 0.05389901, ..., 0.02514575,
         0.0558489 , 0.0023343 ],
        [0.0091505 , 0.00578711, 0.0394147 , ..., 0.01764691,
         0.00184988, 0.01441992],
        ...,
        [0.00126403, 0.00196403, 0.01890185, ..., 0.01535329,
         0.00589   , 0.0198622 ],
        [0.00644403, 0.01326847, 0.01312449, ..., 0.01637871,
         0.0302675 , 0.01438325],
        [0.00090479, 0.01660222, 0.01250244, ..., 0.02528149,
         0.0235401 , 0.0472075 ]],

       [[0.02394231, 0.00634023, 0.02098232, ..., 0.00148407,
         0.01565003, 0.00682806],
        [0.02017674, 0.00667868, 0.00911413, ..., 0.01881391,
         0.00020947, 0.05868836],
        [0.01513833, 0.00658749, 0.01939019, ..., 0.00709752,
         0.0178661 , 0.00055389],
        ...,
        [0.00636817, 0.01042956, 0.00388892, ..., 0.01729405,
         0.01001124, 0.00752211],
        [0.0

: 